In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
insurance = pd.read_csv("insurance.csv")

insurance.head()

In [ ]:
insurance_long = insurance.melt(
    id_vars="charges",
    value_vars=["age", "bmi", "children"],
    var_name="predictor",
    value_name="value"
)

fig = px.scatter(
    insurance_long,
    x="value",
    y="charges",
    facet_col="predictor",
    facet_col_wrap=3,
    trendline="ols",
    opacity=0.5,
    title="Exploring Linear Relationships with Insurance Charges",
    labels={
        "value": "Predictor Value",
        "charges": "Insurance Charges ($)"
    }
)

fig.update_xaxes(matches=None)

fig.show()

In [ ]:
y = insurance["charges"]

X = insurance[["age","bmi","children"]]
Z = insurance[["age","bmi","children", "charges"]]

In [ ]:
corr = Z.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix of Predictors"
)

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    coloraxis_colorbar_title="Correlation"
)

fig.show()

In [ ]:
import statsmodels.api as sm

In [ ]:
X_des = sm.add_constant(X)

X_des.head()

In [ ]:
X_matrix = X_des.to_numpy(dtype=float)
y_vector = y.to_numpy(dtype=float)

beta_hat = (
    np.linalg.inv(X_matrix.T @ X_matrix)
    @ X_matrix.T
    @ y_vector
)

beta_hat

In [ ]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression()

ols.fit(X, y)

In [ ]:
print("Intercept:", ols.intercept_)
print("Coefficients:", ols.coef_)

In [ ]:
original_coefs = pd.Series(
    [
        ols.intercept_,
        *ols.coef_
    ],
    index=[
        "Intercept",
        "age",
        "bmi",
        "children"
    ]
)

original_coefs

In [ ]:
np.random.seed(6021)

B = 1000
n = len(X)

bootstrap_coefs = []

for b in range(B):

    bootstrap_index = np.random.choice(
        n,
        size=n,
        replace=True
    )

    X_boot = X.iloc[bootstrap_index]
    y_boot = y.iloc[bootstrap_index]


    model_boot = LinearRegression()
    model_boot.fit(X_boot, y_boot)


    bootstrap_coefs.append([
        model_boot.intercept_,
        *model_boot.coef_
    ])

bootstrap_coefs = pd.DataFrame(
    bootstrap_coefs,
    columns=[
        "Intercept",
        "age",
        "bmi",
        "children"
    ]
)

bootstrap_coefs.head()

In [ ]:
bootstrap_summary = pd.DataFrame({
    "OLS Estimate": original_coefs,
    "Bootstrap Mean": bootstrap_coefs.mean(),
    "Bootstrap SE": bootstrap_coefs.std()
})

bootstrap_summary.round(2)

In [ ]:
fig = px.histogram(
    bootstrap_coefs,
    x="Intercept",
    nbins=30,
    title="Bootstrap Sampling Distribution of the Intercept",
    labels={"Intercept": "Bootstrap Estimate of Intercept"}
)

fig.add_vline(
    x=original_coefs["Intercept"],
    line_dash="dash"
)

fig.show()

In [ ]:
fig = px.histogram(
    bootstrap_coefs,
    x="age",
    nbins=30,
    title="Bootstrap Sampling Distribution of the Age Coefficient",
    labels={"age": "Bootstrap Estimate of Age Coefficient"}
)

fig.add_vline(
    x=original_coefs["age"],
    line_dash="dash"
)

fig.show()

In [ ]:
fig = px.histogram(
    bootstrap_coefs,
    x="bmi",
    nbins=30,
    title="Bootstrap Sampling Distribution of the BMI Coefficient",
    labels={"bmi": "Bootstrap Estimate of BMI Coefficient"}
)

fig.add_vline(
    x=original_coefs["bmi"],
    line_dash="dash"
)

fig.show()

In [ ]:
bootstrap_ci = pd.DataFrame({
    "Estimate": original_coefs,
    "Bootstrap SE": bootstrap_coefs.std(),
    "Lower 2.5%": bootstrap_coefs.quantile(0.025),
    "Upper 97.5%": bootstrap_coefs.quantile(0.975)
})

bootstrap_ci